# Fig. 4 and Sup. Fig. 10: GSDensity base scores of the breast cancer slides

Per-spot GSDensity relevance (MCA co-embedding, 300-NN graph, random walk with restart; nn_use 300, restart 0.75, MCA
dimensions 1-10, seed 0) of every signature of 09_00_G0_arrest_in_breast_cancer and 09_01_HRD_in_breast_cancer, run per
slide in the R environment gsdensity_env. GSDensity has no gene weights and no signed mode, so G0 arrest and HRD are run
as an up and a down set. One CSV per signature is written to data/gsd_cache/breast/ for the figure notebooks.

In [1]:
import os
import warnings

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import json
import time

from scipy.sparse import issparse

In [3]:
sc.set_figure_params(frameon=False, fontsize=8)
sc.settings._vector_friendly = False
plt.rcParams["axes.grid"] = False

In [4]:
# set working directory
# repository root (the folder containing `notebooks/`); safe to re-run this cell
import sys
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
sys.path.insert(0, project_dir + "notebooks")  # helpers.py and score_signatures.py
os.makedirs(project_dir + "logs", exist_ok=True)
working_dir = project_dir
os.chdir(working_dir)

# set figure directory
figure_dir = working_dir + "figures/"
os.makedirs(figure_dir, exist_ok=True)

# processed data directory
processed_data = project_dir + "processed_data/"

# source data directory
source_dir = working_dir + "source_data/"


# GSDensity scores of this dataset (one CSV per signature) and the per-slide R task directories
gsd_dir = working_dir + "data/gsd_cache/breast/"
task_root = gsd_dir + "_tasks/"

In [5]:
from helpers import (
    present_in_data,
    export_gsdensity_task,
    run_gsdensity,
    read_gsdensity_outputs,
)

In [6]:
# 1. Load data (09_00 / 09_01 cells 5-6)
adata = sc.read_h5ad(processed_data + "adata_slides_0_3.h5ad")
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.obs["batch"] = adata.obs["batch"].astype(str)

slides = ["0", "3"]
patients = {"0": "Patient 1 (Luminal B)", "3": "Patient 2 (Her2+)"}
var = pd.Index(adata.var_names)
print(f"{adata.n_obs} spots x {adata.n_vars} genes")
print(adata.obs["batch"].value_counts().loc[slides].to_string())

5870 spots x 14664 genes
batch
0    2074
3    3796


In [7]:
# 2. Gene sets: G0 arrest and proliferation (09_00 cell 7), hallmarks (08_cancer_hallmarks), HRD (09_01 cell 8)
upregulated_genes = [
    "CFLAR", "CALCOCO1", "YPEL3", "CST3", "SERINC1", "CLIP4", "PCYOX1", "TMEM59","RGS2", "YPEL5", "CD63", "KIAA1109",
    "CDH13", "GSN", "MR1", "CYB5R1", "AZGP1","ZFYVE1", "DMXL1", "EPS8L2", "PTTG1IP", "MIR22HG", "PSAP", "GOLGA8B",
    "NEAT1", "TXNIP", "MTRNR2L12"
]

downregulated_genes = [
    "NCAPD2", "PTBP1", "MPHOSPH9", "NUCKS1", "TCOF1", "SART3", "SNRPA", "KIF22", "HSP90AA1", "WBP11", "CAD", "SF3B2",
    "KHSRP", "WDR76", "NUP188", "HSP90AB1", "HNRNPM", "SMARCB1", "PNN", "RBBP7", "NPRL3", "USP10", "SGTA", "MRPL4",
    "PSMD3", "KPNB1", "CBX1", "LRRC59", "TMEM97", "NSD2", "PRPF19", "PTGES3", "CPSF6", "SRSF3", "TCERG1", "SMC4",
    "EIF4G1", "ZNF142", "MSH6", "MRPL37", "SFPQ", "STMN1", "ARID1A", "PROSER1", "DDX39A", "EXOSC9", "USP22", "DEK",
    "DUT", "ILF3", "DNMT1", "NASP", "HMGB1P5", "SRRM1", "GNL2", "RNF138", "SRSF1", "TRA2B", "SMPD4", "ANP32B", "HMGA1",
    "MDC1", "HADH", "ARHGDIA", "PRCC", "HDGF", "SF3B4", "UBAP2L", "ILF2", "PARP1", "LBR", "CNOT9", "PPRC1", "SSRP1",
    "CCT5", "DLAT", "HNRNPU", "LARP1", "SCAF4", "RRP1B", "RRP1", "CHCHD4", "GMPS", "RFC4", "SLBP", "PSIP1", "HNRNPK",
    "SKA3", "DIS3L", "USP39", "GPS1", "PA2G4", "HCFC1", "SLC19A1", "ETV4", "RAD23A", "DCTPP1", "RCC1", "EWSR1", "ALYREF",
    "PTMA", "HMGB1", "POM121", "MCMBP", "TEAD4", "CHAMP1", "TOP1", "PRRC2A", "RBM14", "HMGB1P6", "POM121C", "UHRF1"
]
proliferation = ["MKI67", "PCNA", "CCND1", "CCNE1", "CCNA1", "CCNB1", "MCM2", "MCM3", "MCM4", "MCM5", "MCM6", "MCM7", "H3F3A", "H3F3B"]

with open(project_dir + "hallmarks_genelists.pkl", "rb") as f:
    cancer_hallmark_genesets = pickle.load(f)

df = pd.read_csv(project_dir + "signature_all_phenotypes.csv", index_col=0)
hrd_all = df["HRD"].dropna()

# distinct gene sets run through GSDensity, as given (before matching to the data)
given_sets = {
    "G0_up": list(upregulated_genes),
    "G0_down": list(downregulated_genes),
    "proliferation": list(proliferation),
    **{h: list(g) for h, g in cancer_hallmark_genesets.items()},
    "HRD_up": list(hrd_all.index[hrd_all > 0]),
    "HRD_down": list(hrd_all.index[hrd_all < 0]),
}

gsd_sets = {k: present_in_data(v, var) for k, v in given_sets.items()}
print(pd.DataFrame({
    "given": {k: len(v) for k, v in given_sets.items()},
    "unique given": {k: len(set(v)) for k, v in given_sets.items()},
    "in data": {k: len(v) for k, v in gsd_sets.items()},
}).to_string())

               given  unique given  in data
G0_up             27            27       23
G0_down          112           112      103
proliferation     14            14       14
H1              1332          1332     1144
H2               580           580      497
H3               237           237      216
H4               146           146      115
H5               661           661      578
H6               683           683      623
H7              1210          1210     1010
H8               321           321      274
H9               235           235      186
H10             1527          1527     1232
H11              270           270      216
H12              350           350      288
H13              411           411      341
HRD_up           131           131      124
HRD_down          97            97       95


In [8]:
# 3. Signature names of the figure notebooks -> GSDensity gene set (signature, gene set run, direction, note)
signature_map = [
    ("G0", "G0_up", "up", "GSDensity score of the signed G0 arrest signature (up set = the 23 listed Wiecek up genes in "
     "the data, whatever sign EnrichMap infers; comparison plots)"),
    ("G0_up", "G0_up", "up", "same file content as G0"),
    ("G0_down", "G0_down", "down", "down set of the signed G0 arrest signature"),
    ("proliferation", "proliferation", "unsigned", "proliferation (Nielsen et al.)"),
    *[(h, h, "unsigned", f"cancer hallmark {h} (Sibai et al.)") for h in cancer_hallmark_genesets],
    ("hrd_gt", "HRD_up", "up", "ground-truth signed Jacobson HRD weights: up set (positive weights; comparison plots)"),
    ("hrd_gt_up", "HRD_up", "up", "same file content as hrd_gt"),
    ("hrd_gt_down", "HRD_down", "down", "ground-truth HRD: down set (negative weights)"),
    ("hrd_pos", "HRD_up", "unsigned", "condition B, positive-weight genes (GSDensity ignores the weights)"),
    ("hrd_inferred", "HRD_up", "unsigned", "inferred weights on the up genes (GSDensity ignores the weights)"),
    ("hrd_inferred_signed", "HRD_up", "up", "signed inferred weights: up set = Jacobson positive-weight genes, as "
     "build_signed_weights is called with them (comparison plots); identical to hrd_gt, GSDensity ignores weights"),
    ("hrd_inferred_signed_up", "HRD_up", "up", "same file content as hrd_inferred_signed"),
    ("hrd_inferred_signed_down", "HRD_down", "down", "signed inferred weights: down set"),
    ("hrd", "HRD_up", "unsigned", "Sup. Fig. 10 comparator signature {'hrd': up genes} (score_signatures.py)"),
]
signature_map = pd.DataFrame(signature_map, columns=["signature", "gene_set", "direction", "note"])
print(signature_map[["signature", "gene_set", "direction"]].to_string(index=False))

               signature      gene_set direction
                      G0         G0_up        up
                   G0_up         G0_up        up
                 G0_down       G0_down      down
           proliferation proliferation  unsigned
                      H1            H1  unsigned
                      H2            H2  unsigned
                      H3            H3  unsigned
                      H4            H4  unsigned
                      H5            H5  unsigned
                      H6            H6  unsigned
                      H7            H7  unsigned
                      H8            H8  unsigned
                      H9            H9  unsigned
                     H10           H10  unsigned
                     H11           H11  unsigned
                     H12           H12  unsigned
                     H13           H13  unsigned
                  hrd_gt        HRD_up        up
               hrd_gt_up        HRD_up        up
             hrd_gt_

In [9]:
# 4. GSDensity bridge: R package in the conda environment gsdensity_env, run by scripts/run_gsdensity.R
gsd_rscript = working_dir + "scripts/run_gsdensity.R"

In [10]:
# 5. Export one task per slide (all gene sets share the slide's MCA embedding and graph)
gsd_params = {"nn_use": 300, "restart": 0.75, "dims": 10, "seed": 0, "spatial": True, "spatial_n": 10,
              "spatial_ntimes": 100}
task_dirs = {b: task_root + f"slide_{b}/" for b in slides}
for b in slides:
    sub = adata[adata.obs["batch"] == b].copy()
    export_gsdensity_task(sub, gsd_sets, task_dirs[b], gsd_params)
    print(f"slide {b}: exported {sub.n_obs} spots x {sub.n_vars} genes -> {task_dirs[b]}")

slide 0: exported 2074 spots x 14664 genes -> /home/cenk/Celik_enrichmap/revision/data/gsd_cache/breast/_tasks/slide_0/


slide 3: exported 3796 spots x 14664 genes -> /home/cenk/Celik_enrichmap/revision/data/gsd_cache/breast/_tasks/slide_3/


In [11]:
# 6. Run GSDensity (the two slides in two parallel R sessions)
t0 = time.time()
rc = run_gsdensity(list(task_dirs.values()), gsd_rscript, n_jobs=2,
                   log=working_dir + "logs/REV_GSD_base_breast_R.log")
wall_s = time.time() - t0
print(f"R exit codes: {rc}; wall clock: {wall_s:.1f} s")
for i in range(2):
    print("".join(l for l in open(working_dir + f"logs/REV_GSD_base_breast_R_{i}.log")
                  if l.startswith("[gsdensity]")))

R exit codes: [0, 0]; wall clock: 154.7 s
[gsdensity] /home/cenk/Celik_enrichmap/revision/data/gsd_cache/breast/_tasks/slide_0/ done in 93.4s (mca 25.8s, rwr 18.1s)

[gsdensity] /home/cenk/Celik_enrichmap/revision/data/gsd_cache/breast/_tasks/slide_3/ done in 134.3s (mca 34.5s, rwr 20.9s)



In [12]:
# 7. Read the per-slide results: sum of relevance, antimode-positive spots and NA labels per gene set
gsd = {b: read_gsdensity_outputs(task_dirs[b]) for b in slides}

check_rows = []
for b in slides:
    scores_b, labels_b = gsd[b]["scores"], gsd[b]["labels"]
    for k in gsd_sets:
        check_rows.append({
            "slide": b,
            "gene_set": k,
            "n_spots": len(scores_b),
            "sum relevance": scores_b[k].sum(),
            "n positive (antimode)": int((labels_b[k] == "positive").sum()),
            "label NA": int(labels_b[k].isna().sum()),
        })
checks = pd.DataFrame(check_rows)
print(checks.pivot(index="gene_set", columns="slide", values=["n positive (antimode)", "label NA"])
      .loc[list(gsd_sets)].to_string())

              n positive (antimode)       label NA      
slide                             0     3        0     3
gene_set                                                
G0_up                           153    97        0     0
G0_down                          34     2        0     0
proliferation                    10   103        0     0
H1                             1655     0        0  3796
H2                                7     1        0     0
H3                                0     1     2074     0
H4                               45     3        0     0
H5                                1     4        0     0
H6                              770     2        0     0
H7                             1627     0        0  3796
H8                                1     2        0     0
H9                             1344  2432        0     0
H10                            1585  1949        0     0
H11                              51     1        0     0
H12                            

In [13]:
# 8. Genes used per slide (MCA drops genes with zero variance in the slide; RWR seeds = set genes in the graph)
genes_used = {}
for b in slides:
    X = adata[adata.obs["batch"] == b].X
    X = X.tocsc() if issparse(X) else X
    mean = np.asarray(X.mean(axis=0)).ravel()
    sq = np.asarray(X.multiply(X).mean(axis=0)).ravel() if issparse(X) else (X**2).mean(axis=0)
    varying = set(var[(sq - mean**2) > 0])
    genes_used[b] = {k: [g for g in v if g in varying] for k, v in gsd_sets.items()}
used_table = pd.DataFrame({f"slide {b}": {k: len(v) for k, v in genes_used[b].items()} for b in slides})
used_table.insert(0, "in data", {k: len(v) for k, v in gsd_sets.items()})
used_table.insert(0, "given", {k: len(set(v)) for k, v in given_sets.items()})
print(used_table.to_string())

               given  in data  slide 0  slide 3
G0_up             27       23       23       23
G0_down          112      103      103      103
proliferation     14       14       14       14
H1              1332     1144     1139     1141
H2               580      497      495      497
H3               237      216      216      216
H4               146      115      115      115
H5               661      578      577      577
H6               683      623      619      621
H7              1210     1010     1001     1006
H8               321      274      274      274
H9               235      186      185      184
H10             1527     1232     1218     1227
H11              270      216      216      216
H12              350      288      288      288
H13              411      341      334      339
HRD_up           131      124      124      124
HRD_down          97       95       95       94


Two native properties of GSDensity are kept as it produces them and counted in the index: `dnet::dRWR` sets
steady-state probabilities below 1e-6 to zero, so many spots have relevance exactly 0 (ties), and `compute.cell.label`
returns NA for every spot of a slide when `multimode::locmodes` finds one mode only (no antimode).

In [14]:
# 9. One CSV per signature (spot_id, slide, relevance, antimode label, relevance x spots in the slide) and the index
timing = {b: gsd[b]["timing"] for b in slides}
runtime_total = sum(t["total"] for t in timing.values())
runtime_note = "; ".join(
    f"slide {b}: total {t['total']:.1f} s (load {t['load']:.1f}, MCA {t['mca']:.1f}, kNN {t['nn']:.1f}, "
    f"RWR {t['rwr']:.1f}, spatial KLD {t.get('spatial_kld', 0):.1f})"
    for b, t in timing.items()
) + f"; one run per slide shared by all {len(gsd_sets)} gene sets (runtime is not per signature)"
n_spots = {b: int((adata.obs["batch"] == b).sum()) for b in slides}
subsample_note = (
    f"none: all {adata.n_obs} spots; GSDensity run per slide (slide 0: {n_spots['0']} spots, slide 3: "
    f"{n_spots['3']} spots), one MCA embedding per slide; relevance sums to 1 within each slide"
)

index_rows = []
for _, r in signature_map.iterrows():
    parts = []
    for b in slides:
        s = gsd[b]["scores"][r.gene_set]
        parts.append(pd.DataFrame({
            "spot_id": s.index,
            "slide": b,
            "relevance": s.values,
            "label": gsd[b]["labels"].loc[s.index, r.gene_set].values,
            "relevance_scaled": s.values * len(s),
        }))
    tab = pd.concat(parts, ignore_index=True).set_index("spot_id").reindex(adata.obs_names)
    tab.index.name = "spot_id"
    tab = tab.reset_index()
    tab.to_csv(gsd_dir + f"{r.signature}.csv", index=False)

    na_slides = [b for b in slides if tab.loc[tab["slide"] == b, "label"].isna().all()]
    zero = {b: int((tab.loc[tab["slide"] == b, "relevance"] == 0).sum()) for b in slides}
    n_used = min(len(genes_used[b][r.gene_set]) for b in slides)
    index_rows.append({
        "dataset": "breast",
        "signature": r.signature,
        "gene_set_run": r.gene_set,
        "direction": r.direction,
        "genes_used": n_used,
        "genes_used_of_given": f"{n_used}/{len(set(given_sets[r.gene_set]))}",
        "genes_given": len(set(given_sets[r.gene_set])),
        "genes_in_data": len(gsd_sets[r.gene_set]),
        "genes_used_per_slide": "; ".join(f"slide {b}: {len(genes_used[b][r.gene_set])}" for b in slides),
        "n_spots": len(tab),
        "n_positive_label": int((tab["label"] == "positive").sum()),
        "n_label_NA": int(tab["label"].isna().sum()),
        "label_note": (
            "label NA for all spots of slide " + " and ".join(na_slides) + ": compute.cell.label found one mode only "
            "(multimode::locmodes gives no antimode); native GSDensity behaviour"
        ) if na_slides else "",
        "n_zero_relevance": sum(zero.values()),
        "zero_relevance_note": (
            "; ".join(f"slide {b}: {zero[b]}/{n_spots[b]}" for b in slides)
            + " spots with relevance exactly 0 (dnet::dRWR sets probabilities < 1e-6 to 0; ties)"
        ),
        "subsample_note": subsample_note,
        "runtime_s": round(runtime_total, 1),
        "runtime_note": runtime_note,
        "params": json.dumps(gsd_params),
        "note": r.note,
        "file": f"data/gsd_cache/breast/{r.signature}.csv",
    })
index = pd.DataFrame(index_rows)
index.to_csv(working_dir + "data/gsd_cache/INDEX_breast.csv", index=False)
print(index[["signature", "gene_set_run", "direction", "genes_used", "genes_given", "n_spots", "n_positive_label",
             "n_label_NA", "n_zero_relevance", "file"]].to_string(index=False))

               signature  gene_set_run direction  genes_used  genes_given  n_spots  n_positive_label  n_label_NA  n_zero_relevance                                               file
                      G0         G0_up        up          23           27     5870               250           0              4187                       data/gsd_cache/breast/G0.csv
                   G0_up         G0_up        up          23           27     5870               250           0              4187                    data/gsd_cache/breast/G0_up.csv
                 G0_down       G0_down      down         103          112     5870                36           0              5431                  data/gsd_cache/breast/G0_down.csv
           proliferation proliferation  unsigned          14           14     5870               113           0              4757            data/gsd_cache/breast/proliferation.csv
                      H1            H1  unsigned        1139         1332     5870        

In [15]:
# 10. Pathway-level spatial KLD per slide (compute.spatial.kld, 10 x 10 grid, 100 spot shuffles)
kld_rows = []
for b in slides:
    sk = gsd[b]["spatial_kld"]
    for k in gsd_sets:
        kld_rows.append({"slide": b, "patient": patients[b], "gene_set": k, **sk.loc[k].to_dict()})
kld = pd.DataFrame(kld_rows)
print(kld.pivot(index="gene_set", columns="slide", values="delta.kld").loc[list(gsd_sets)].round(3).to_string())

slide              0      3
gene_set                   
G0_up          1.692  1.901
G0_down        2.014  1.128
proliferation  2.074  1.587
H1               NaN  3.679
H2             3.017  3.496
H3             2.596  3.052
H4             2.476  2.511
H5             2.802  3.482
H6             2.597  3.637
H7               NaN  3.708
H8             2.531  2.489
H9               NaN  3.682
H10              NaN  3.942
H11            2.612  3.179
H12            2.604  3.141
H13              NaN  3.522
HRD_up         2.756  2.917
HRD_down       2.784  3.405


In [16]:
# 11. Summary per gene set
pos = checks.pivot(index="gene_set", columns="slide", values="n positive (antimode)")
padj = kld.pivot(index="gene_set", columns="slide", values="p.adj")
summary = pd.DataFrame({
    "genes given": used_table["given"],
    "genes used (slide 0 / 3)": [f"{used_table.loc[k, 'slide 0']} / {used_table.loc[k, 'slide 3']}" for k in used_table.index],
    "positive spots (slide 0 / 3)": [f"{pos.loc[k, '0']} / {pos.loc[k, '3']}" for k in used_table.index],
    "spatial KLD p.adj (slide 0 / 3)": [f"{padj.loc[k, '0']:.1e} / {padj.loc[k, '3']:.1e}" for k in used_table.index],
    "zero relevance (slide 0 / 3)": [
        f"{(gsd['0']['scores'][k] == 0).mean():.0%} / {(gsd['3']['scores'][k] == 0).mean():.0%}" for k in used_table.index
    ],
    "label NA (slide 0 / 3)": [
        f"{int(gsd['0']['labels'][k].isna().sum())} / {int(gsd['3']['labels'][k].isna().sum())}" for k in used_table.index
    ],
})
print(summary.to_string())
print(f"{len(signature_map)} signature files from {len(gsd_sets)} GSDensity gene sets, {adata.n_obs} spots; "
      f"GSDensity runtime {runtime_total:.1f} s summed over the two slides ({wall_s:.1f} s wall clock)")

               genes given genes used (slide 0 / 3) positive spots (slide 0 / 3) spatial KLD p.adj (slide 0 / 3) zero relevance (slide 0 / 3) label NA (slide 0 / 3)
G0_up                   27                  23 / 23                     153 / 97               4.0e-13 / 4.3e-26                    48% / 84%                  0 / 0
G0_down                112                103 / 103                       34 / 2               3.5e-18 / 1.7e-11                    82% / 98%                  0 / 0
proliferation           14                  14 / 14                     10 / 103               6.2e-34 / 2.6e-16                    66% / 89%                  0 / 0
H1                    1332              1139 / 1141                     1655 / 0                   nan / 2.6e-79                     1% / 35%               0 / 3796
H2                     580                495 / 497                        7 / 1               3.5e-17 / 1.5e-76                    10% / 55%                  0 / 0
H3        